一、导入 Cookie Cats 数据并做首次体检

In [ ]:

#1.1导入数据
import pandas as pd       #数据处理库，负责读 CSV；
from pathlib import Path  #Python 标准库的路径对象，用来做路径拼接、判断文件是否存在，跨平台

# 自动向上查找 data 文件夹，避免相对路径踩坑
p = Path.cwd()                                                          #返回当前工作目录的绝对路径，取决于你从哪个目录启动 Python
while not (p / 'data' / 'cookie_cats.csv').exists() and p != p.parent:  #当前目录下找不到 data/cookie_cats.csv 且还没爬到文件系统根目录
    p = p.parent                                                        #否则返回p（根目录）

DATA = p / 'data' / 'cookie_cats.csv'
print('数据路径：', DATA)
print('文件存在：', DATA.exists())

df = pd.read_csv(DATA)
print('形状（行, 列）：', df.shape) 

df.head(10)   #查看首10行


In [ ]:
#1.2查看数据结构情况
df.info()                            #整体结构信息：每列缺多少、数据类型是否合理、内存占用
print('\n--- 各组人数 ---')           #分组是否均衡？（A/B 测试的前提）
print(df['version'].value_counts())
print('\n--- 缺失值 ---')
print(df.isnull().sum())             #可能检查不全，可用：df.replace(['', 'N/A', '-', 'NULL'], pd.NA).isnull().sum()
print('\n--- 描述统计 ---')
print(df.describe())                 #数值分布什么样？有没有异常值？

结论
1. userid 是编号，describe() 无业务含义（均值 45095 不代表任何东西）。
2. sum_gamerounds 存在极端值（49854 轮），一个用户就能把均值从 16 拉到 52。
   处理决定：不删除（无证据表明是数据错误，删除会引入选择偏差），
   改为主分析用中位数 + Mann-Whitney 秩检验（对极端值免疫），
   并将 99 分位截尾 t 检验作为稳健性对照。


In [ ]:
#1.3 看 sum_gamerounds 的极端值
print('最大值：', df['sum_gamerounds'].max())
print('99 分位：', df['sum_gamerounds'].quantile(0.99))
print('中位数：', df['sum_gamerounds'].median())

# 排序看最极端的 5 个人
df.nlargest(5, 'sum_gamerounds')[['userid','version','sum_gamerounds','retention_1','retention_7']]


In [ ]:
#1.4看两组留存率
df.groupby('version')[['retention_1','retention_7']].agg(['mean','count'])


In [ ]:
#1.5'retention_1','retention_7'释义
#'retention_1','retention_7'代表“第 1 （次日）天当天有没有来”、“第 7 天当天有没有来”。所以存在“第 1 天没来、第 7 天来了”的人。
pd.crosstab(df['retention_1'], df['retention_7'], margins=True)
#如果交叉表里出现 retention_1=False & retention_7=True 的行，就说明两个指标不嵌套

二、SRM 与分组平衡性检验

In [ ]:
#2.1 SRM 卡方检验 一般要求p值大于等于0.001 不拒绝原假设 分流可信
from scipy import stats
import numpy as np

obs = np.array([44700, 45489])
exp = np.array([90189/2, 90189/2])
chi2, p = stats.chisquare(obs, exp)
print(f'chi2={chi2:.4f}, p={p:.6f}')


In [ ]:
#2.2平衡性检查
#这个数据集没有任何协变量——没有设备、地区、渠道、国家、安装时间。所以严格意义上，做不了真正的平衡性检查。做两项替代检验

#2.2.1替代检验 A：userid 分布是否随机
#如果分流是按 hash(userid) 做的，userid 的取值分布应该与分组无关。
df.groupby('version')['userid'].describe()


结论：“两组 userid 分布高度一致，与随机分流的特征相符，未发现分流按 ID 段切分的证据”。因为你只能证伪，不能证明。最小值（116 vs 377）的差异不用管，极端值附近本来就是稀疏的。

In [ ]:
#2.2.2替代检验 B：伪 AA 测试
#用 userid % 2 造一个“假分组”，看两组留存率是否无差异 
df['pseudo'] = df['userid'] % 2
summary = df.groupby('pseudo')[['retention_1', 'retention_7']].agg(['mean', 'count'])
print(summary)

# ===== 比例检验：看 p 值 =====
from statsmodels.stats.proportion import proportions_ztest
print('\n=== 伪 AA 测试：两独立比例 z 检验 ===')
for col in ['retention_1', 'retention_7']:
    grp = df.groupby('pseudo')[col].agg(['sum', 'count'])
    successes = grp['sum'].to_numpy()    # 各组留存人数（bool 求和即 True 的个数）
    nobs = grp['count'].to_numpy()       # 各组总人数
    rate = successes / nobs              # 各组留存率

    stat, p = proportions_ztest(successes, nobs)

    print(f'\n{col}:')
    print(f'  组0 留存率 = {rate[0]:.4%}  ({successes[0]}/{nobs[0]})')
    print(f'  组1 留存率 = {rate[1]:.4%}  ({successes[1]}/{nobs[1]})')
    print(f'  组间差值 = {rate[1] - rate[0]:+.4%}')
    print(f'  z = {stat:.4f}, p = {p:.4f}')
    if p >= 0.05:
        print('  → p ≥ 0.05，两组无显著差异（符合 AA 测试预期 ✅）')
    else:
        print('  → p < 0.05，伪分组竟然显著！需排查分组/指标/数据 ⚠️')

结论：不显著 → 伪 AA 通过 → 说明后续主分析的结果不会被“系统性偏差”污染。  
补充一个有效性判断：pseudo=0 有 44,966 人、pseudo=1 有 45,223 人，而 version 是 44,700 / 45,489，两组人数不同，  
说明 userid % 2 和 version 不是同一个划分方式，这个测试是有效的（如果两者完全等价，测试就失效了）。

三、主分析——留存检验、置信区间、效应量

In [ ]:
#3.1分析主代码：比例 → z 检验 → 置信区间 → 效应量
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep  #两独立比例 z 检验、两比例之差的置信区间
from scipy.stats import chi2_contingency  #卡方检验

g30 = df[df['version'] == 'gate_30']
g40 = df[df['version'] == 'gate_40']

rows = []
for col in ['retention_1', 'retention_7']:
    x1, n1 = int(g30[col].sum()), len(g30)   # gate_30 = 对照组，（留存人数，组内总人数）
    x2, n2 = int(g40[col].sum()), len(g40)   # gate_40 = 实验组
    p1, p2 = x1/n1, x2/n2                    # 两组留存率

    z, p = proportions_ztest([x1, x2], [n1, n2])                            # 独立比例 z 检验， H0: p1 = p2（合并方差版本）
    lo, hi = confint_proportions_2indep(x1, n1, x2, n2, method='wald')      # p1 − p2 的 95% 置信区间（Wald 法）
    h = 2*np.arcsin(np.sqrt(p1)) - 2*np.arcsin(np.sqrt(p2))                 # Cohen's h 效应量

    rows.append({
        '指标': col,
        'gate_30': round(p1*100, 2), 'gate_40': round(p2*100, 2),
        '绝对差(pp)': round((p1-p2)*100, 2),
        '相对差(%)': round((p1-p2)/p1*100, 2),
        'z': round(z, 3), 'p值': round(p, 4),
        'CI下界(pp)': round(lo*100, 2), 'CI上界(pp)': round(hi*100, 2),
        "Cohen's h": round(h, 3)
    })

result = pd.DataFrame(rows)             #转成表
print(result.to_string(index=False))    #打印时去掉左侧行号、列对齐


结论：结论先行 → 数字支撑 → 证据强度 → 限定条件  
30 字版（给产品经理，口头说）​

40 关方案 7 日留存低 0.82pp，区间不跨 0，建议不上线。

60 字版（周报/群消息）​

门槛后移到 40 关使 7 日留存从 19.02% 降至 18.20%（-0.82pp，相对 -4.31%，p=0.0016，95%CI [0.31, 1.33]pp），方向稳健；效应量虽小，但按日增 10 万测算年损失约 30 万留存用户，建议维持 30 关。

完整版（报告结论）​

主指标显著为负且置信区间完全位于 0 以上，证据强度足以支持“不上线”的决策；但效应量极小（Cohen's h=0.021），说明问题不在“门槛位置”本身，而在 31–40 关的内容承接能力，建议后续优先优化该区间的难度曲线与奖励节奏，而非调整门槛位置。

In [ ]:
#3.2用卡方检验交叉验证
for col in ['retention_1', 'retention_7']:
    table = pd.crosstab(df['version'], df[col])                     #构造 2×2 列联表，crosstab（交叉表）统计"行取值 × 列取值"的频数。卡方检验的标准输入：观测频数表
    chi2, p, dof, exp = chi2_contingency(table, correction=False)   #卡方检验
    print(f'{col}: chi2={chi2:.4f}, p={p:.6f}')
    print(f'  验证: z^2 = {result.loc[result.指标==col, "z"].values[0]**2:.4f}（应约等于 chi2）')


结论：  
2×2 表双侧检验下 z² = χ² 成立，说明检验口径与代码一致。

四、轮次分析与稳健性检验  
轮次分析：对连续型参与度指标“游戏轮次”做组间比较，回答“新方案让用户用得更深还是更浅”
稳健性检验：用多种方法、多种口径、多个子样本重复同一个结论，看结论会不会变。
处理那个 49,854 轮的极端值，用两种方法证明结论不依赖它。

In [ ]:
#4.1描述统计，先暴露“右偏”
desc = df.groupby('version')['sum_gamerounds'].describe(
    percentiles=[.25, .5, .75, .9, .99]
)
print(desc)
print('\n均值 vs 中位数：')
print(df.groupby('version')['sum_gamerounds'].agg(['mean','median']))


结论：  
均值约 51.298776，中位数约 16，均值是中位数的 3 倍多——典型的右偏分布；  
标准差是 2.5 倍， 直接破坏了方差的可比性，而 t 检验恰恰要用方差。

In [ ]:
#4.2Mann-Whitney U（对极端值免疫）
#非参数统计：秩检验——只看"谁排前面"，天然抗离群值
#严格的 H0 是"两组分布相同（随机优势为 0.5）"。只有当两组的分布形状相近时，才能把它解释成"中位数/位置有差异"。
from scipy.stats import mannwhitneyu

x30 = g30['sum_gamerounds']
x40 = g40['sum_gamerounds']
u, p = mannwhitneyu(x30, x40, alternative='two-sided')
print(f'Mann-Whitney U = {u:.1f}, p = {p:.6f}')

# 效应量：秩二列相关 r = Z / sqrt(N)
import numpy as np
from scipy.stats import norm
n1, n2 = len(x30), len(x40)
z_stat = norm.isf(p / 2)
sign = 1 if u > n1 * n2 / 2 else -1     # U 大于期望 → x30 整体更大
r = sign * z_stat / np.sqrt(n1 + n2)
print(f'效应量 r = {r:.4f}')             # 负值表示实验组玩得更多


结论：  
统计层：p = 0.0502 > 0.05，未能在 5% 水平拒绝 H0，证据不足以认定两组分布不同；但 p 值紧贴阈值，属于"边缘情况"，提示可能存在极微弱差异。  
业务层：效应量 r = 0.0065，几乎为零（|r| < 0.1 属可忽略）。所以"即使存在差异，也小到没有业务意义"

In [ ]:
#4.3截尾后做 t 检验（稳健性对照）
#对比“不截尾 t 检验”和“截尾后 t 检验”的 p 值。如果两者差异很大，说明极端值确实在主导均值，那你就更有理由主用 Mann-Whitney。
from scipy.stats import ttest_ind

cap = df['sum_gamerounds'].quantile(0.99)   # 99 分位截尾
print('截尾阈值：', cap)

df['gr_cap'] = df['sum_gamerounds'].clip(upper=cap)
t, p_t = ttest_ind(
    df.loc[df.version=='gate_30','gr_cap'],
    df.loc[df.version=='gate_40','gr_cap'],
    equal_var=False
)
print(f'截尾后 t = {t:.4f}, p = {p_t:.6f}')
print('截尾后均值：')
print(df.groupby('version')['gr_cap'].mean())

# 对照：不截尾的 t 检验（看极端值把结论带偏了多少）
t_raw, p_raw = ttest_ind(x30, x40, equal_var=False)
print(f'不截尾 t = {t_raw:.4f}, p = {p_raw:.6f}')


结论：  
1.截尾后 p 变大（0.615 > 0.376），说明极端值确实在"制造"均值差异。更准确地说：极端值同时抬高了 gate_30 的均值和方差，两个作用部分抵消，但净效果是让 t 统计量的分母变得不可靠。  
2.t检验问"均值是否不同"——被极端值污染；MW 问"整条分布的位置是否不同"（基于秩）——对 17 vs 16 这种整体右移很敏感，对个别极端值免疫。  
3.三种方法都指向"轮次无显著差异"，​轮次指标不支持"gate_40 让玩家玩得更多"的假设。


In [ ]:
#4.4画图
# ========== Cell 1：统一样式 ==========
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']   # macOS 上兼容性最好的中文字体
plt.rcParams['axes.unicode_minus'] = False   # 负号正常显示
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'       # 保存时不裁掉标签

# --- 定位 images 目录（用 data 反查项目根，避免建错位置）---
PROJ = Path.cwd()
while not (PROJ / 'data' / 'cookie_cats.csv').exists() and PROJ != PROJ.parent:
    PROJ = PROJ.parent
IMG = PROJ / 'images'
IMG.mkdir(exist_ok=True)
print('图片保存到：', IMG)

# ========== Cell 2：图 1 轮次分布（裁到 ≤500，因为存在极端值）==========
CAP_SHOW = 500
cap99 = df['sum_gamerounds'].quantile(0.99)
n_over = (df['sum_gamerounds'] > CAP_SHOW).sum()

fig, ax = plt.subplots(figsize=(9, 4.2))
colors = {'gate_30': '#4C78A8', 'gate_40': '#F58518'}

for v in ['gate_30', 'gate_40']:
    s = df.loc[(df['version'] == v) & (df['sum_gamerounds'] <= CAP_SHOW),
               'sum_gamerounds']
    ax.hist(s, bins=np.arange(0, CAP_SHOW + 10, 10),
            alpha=0.55, label=f'{v}（n={len(s):,}，中位数 {s.median():.0f}）',
            color=colors[v], edgecolor='white', linewidth=0.3)

ax.set_xlabel('安装后 14 天内的游戏轮次')
ax.set_ylabel('玩家数')
ax.set_title(f'两组游戏轮次分布（仅显示 ≤{CAP_SHOW} 轮，占全部 {1-n_over/len(df):.2%}；'
             f'另有 {n_over} 人超过该值，最高 49,854 轮）',
             fontsize=11)
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.savefig(IMG / 'gamerounds_dist.png')
plt.show()

# ========== Cell 3：图 2 CDF 对比 ==========
fig, ax = plt.subplots(figsize=(7, 4.2))

for v in ['gate_30', 'gate_40']:
    s = np.sort(df.loc[df['version'] == v, 'sum_gamerounds']
                .clip(upper=cap99).values)
    ax.plot(s, np.arange(1, len(s) + 1) / len(s),
            label=v, color=colors[v], lw=1.8)

ax.set_xlim(0, 300)          # 300 轮已覆盖 95% 以上用户，长尾不占版面
ax.set_ylim(0, 1.02)
ax.set_xlabel(f'游戏轮次（已截尾至 99 分位 {cap99:.0f} 轮）')
ax.set_ylabel('累积占比')
ax.set_title('两组轮次累积分布：两条曲线高度重合 → 分布无实质差异')
ax.legend(frameon=False, loc='lower right')
ax.grid(alpha=0.25)
plt.savefig(IMG / 'cdf_compare.png')
plt.show()

# ========== Cell 4：图 3 留存差异 + 95%CI =========
from statsmodels.stats.proportion import confint_proportions_2indep

g30 = df[df['version'] == 'gate_30']
g40 = df[df['version'] == 'gate_40']

rows = []
for col, label in [('retention_7', '7 日留存（主指标）'),
                   ('retention_1', '1 日留存（护栏）')]:
    x1, n1 = int(g30[col].sum()), len(g30)
    x2, n2 = int(g40[col].sum()), len(g40)
    d = x1/n1 - x2/n2
    lo, hi = confint_proportions_2indep(x1, n1, x2, n2, method='wald')
    # 转成百分点
    rows.append((label, d*100, lo*100, hi*100))

fig, ax = plt.subplots(figsize=(8, 2.9))
for i, (label, d, lo, hi) in enumerate(rows):
    ax.errorbar(d, i, xerr=[[d - lo], [hi - d]], fmt='o',
                color='#C44E52', capsize=6, markersize=7, lw=1.8)
    ax.text(hi + 0.08, i,
            f'{d:+.2f}pp  95%CI [{lo:+.2f}, {hi:+.2f}]',
            va='center', fontsize=9)

ax.axvline(0, ls='--', color='gray', lw=1.2)
ax.set_yticks(range(len(rows)))
ax.set_yticklabels([r[0] for r in rows])
ax.set_xlim(-0.6, 2.6)
ax.set_xlabel('gate_30 − gate_40 的差异（百分点）｜正值 = gate_40 更差')
ax.set_title('留存指标差异及 95% 置信区间')
ax.grid(axis='x', alpha=0.25)
plt.savefig(IMG / 'retention_ci.png')
plt.show()

print('图 3 数据核对：', rows)


In [ ]:
#4.5分层分析
#目的：按 userid 分组（userid 大致对应安装时间先后），用来检验效应是否随时间衰减（新奇效应）
df['uid_q'] = pd.qcut(df['userid'], 4, labels=['Q1(最早)','Q2','Q3','Q4(最晚)'])

for q in ['Q1(最早)','Q2','Q3','Q4(最晚)']:
    sub = df[df.uid_q == q]
    r30 = sub[sub.version=='gate_30']['retention_7'].mean()
    r40 = sub[sub.version=='gate_40']['retention_7'].mean()
    print(f'{q}: gate_30={r30:.4f}, gate_40={r40:.4f}, diff={r30-r40:+.4f}, n={len(sub)}')


结论：  
1.方向一致性：四个安装批次里，gate_40 的 7 日留存无一例外都低于 gate_30。如果结论是某个时间段（比如某次版本更新）造成的，应该只在某一层出现，或方向反转。  
2.幅度稳定性：四层差异在 0.60pp ~ 1.12pp 之间波动，围绕整体值 0.82pp 上下浮动，波动幅度在抽样误差范围内（每层 n≈22,500，MDE 约 1.0pp，所以层间差异不显著是正常的）。  
3.有无趋势：Q1(0.60) → Q2(1.12) → Q3(0.72) → Q4(0.84)，没有单调趋势，说明不存在"效应随时间放大或衰减"的新奇效应。  
4.一句话结论：gate_40 的劣势在所有安装批次中稳定存在，是产品层面的真实效应，不是时间、批次或短期波动造成的。

In [ ]:
#换推断方式--Bootstrap
np.random.seed(42)
n_boot = 5000
diffs = np.empty(n_boot)

s30 = g30['retention_7'].values
s40 = g40['retention_7'].values

for i in range(n_boot):
    b30 = np.random.choice(s30, len(s30), replace=True)
    b40 = np.random.choice(s40, len(s40), replace=True)
    diffs[i] = b30.mean() - b40.mean()

lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f'Bootstrap 95% CI: [{lo*100:+.2f}pp, {hi*100:+.2f}pp]')
print(f'均值: {diffs.mean()*100:+.2f}pp, 标准差: {diffs.std()*100:.3f}pp')


结论：  
Bootstrap 的 CI 和 （主分析）Wald CI [+0.31, +1.33] 非常接近。如果接近，说明结论不依赖于正态近似假设